# Imports

In [1]:
import os
os.environ['KERAS_BACKEND'] = 'tensorflow'

import keras
import numpy as np
import pandas as pd
import transformers
import tensorflow as tf
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

import sentencepiece as spm

I0000 00:00:1795394620.701938  177222 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


# Choose the dataset

Get the dataset from [here](https://tatoeba.org/en/downloads). Preferably use russian to english translations.

In [2]:
dataset = pd.read_csv('dataset.tsv', sep='\t', header=None, usecols=[1, 3])
len(dataset)


830696

In [3]:
dataset.head()

,1,3
0,Один раз в жизни я делаю хорошее дело... И оно...,For once in my life I'm doing a good deed... A...
1,Давайте что-нибудь попробуем!,Let's try something.
2,Мне пора идти спать.,I have to go to sleep.
3,Что ты делаешь?,What are you doing?
4,Что ты делаешь?,What do you make?


In [4]:
dataset[-10:]

,1,3
830686,"Поверить не могу, что уже Рождество.",I can't believe it's Christmas already.
830687,Я ничего не готовила.,I wasn't cooking anything.
830688,Я ничего не готовил.,I wasn't cooking anything.
830689,Я сегодня не буду готовить.,Today I will not cook.
830690,Мне нужна сумка. Можете одолжить?,I need a bag. Will you lend me one?
830691,Мне нужна сумка. Можешь одолжить?,I need a bag. Will you lend me one?
830692,Мне нравятся шахматы.,I like chess.
830693,Она пожаловалась ему на еду.,She complained to him about the food.
830694,Все персонажи легко узнаваемы.,All the characters are easily recognizable.
830695,"«Кто эта женщина в белом платье?» — «Это Мэри,...","Who is the lady in the white dress? ""It's Mary..."


# Tokenizer

Train your own tokenizer on a corpus for the languages you've chosen or load a pretrained tokenizer. The tokenizer must allow BOS, EOS and PAD tokens.

In [5]:
# with open("tokenizer_corpus.txt", "w", encoding="utf-8") as f:
#     for el in tqdm(dataset.iterrows()):
#         f.write(el[1][1].strip() + '\n')
#         f.write(el[1][3].strip() + '\n')
        

In [6]:
tokenizers_folder = 'tokenizers/'

# spm.SentencePieceTrainer.train(
#     input='tokenizer_corpus.txt',
#     model_prefix=tokenizers_folder + 'ru_en_tokenizer8k',

#     model_type='unigram',
#     vocab_size=8000,

#     character_coverage=1.0,
#     byte_fallback=True,

#     pad_id=0,
#     unk_id=1,
#     bos_id=2,
#     eos_id=3,
# )

In [7]:
tokenizer8k = spm.SentencePieceProcessor(
    model_file=tokenizers_folder + 'ru_en_tokenizer8k.model'
)
tokenizer16k = spm.SentencePieceProcessor(
    model_file=tokenizers_folder + 'ru_en_tokenizer16k.model'
)
tokenizer32k = spm.SentencePieceProcessor(
    model_file=tokenizers_folder + 'ru_en_tokenizer32k.model'
)
tokenizer48k = spm.SentencePieceProcessor(
    model_file=tokenizers_folder + 'ru_en_tokenizer48k.model'
)

text = 'Привет, how are you?'

tokens = tokenizer16k.encode(text, out_type=str)
ids = tokenizer16k.encode(text, out_type=int)

print(tokens)
print(ids)

print(tokenizer16k.decode(ids))

['▁Привет', ',', '▁how', '▁are', '▁you', '?']
[5563, 261, 493, 316, 266, 263]
Привет, how are you?


In [8]:
ru_tok_8k = [np.array(tokenizer8k.encode(i, out_type=int)) for i in tqdm(dataset[1])]
en_tok_8k = [np.array(tokenizer8k.encode(i, out_type=int)) for i in tqdm(dataset[3])]

ru_tok_16k = [np.array(tokenizer16k.encode(i, out_type=int)) for i in tqdm(dataset[1])]
en_tok_16k = [np.array(tokenizer16k.encode(i, out_type=int)) for i in tqdm(dataset[3])]

ru_tok_32k = [np.array(tokenizer32k.encode(i, out_type=int)) for i in tqdm(dataset[1])]
en_tok_32k = [np.array(tokenizer32k.encode(i, out_type=int)) for i in tqdm(dataset[3])]

ru_tok_48k = [np.array(tokenizer48k.encode(i, out_type=int)) for i in tqdm(dataset[1])]
en_tok_48k = [np.array(tokenizer48k.encode(i, out_type=int)) for i in tqdm(dataset[3])]

100%|██████████| 830696/830696 [00:04<00:00, 176839.46it/s]


In [9]:
ru_symbol_len = np.array(dataset[1].str.len())
en_symbol_len = np.array(dataset[3].str.len())

ru_tok_8k_len = np.array([len(t) for t in ru_tok_8k])
en_tok_8k_len = np.array([len(t) for t in en_tok_8k])

ru_tok_16k_len = np.array([len(t) for t in ru_tok_16k])
en_tok_16k_len = np.array([len(t) for t in en_tok_16k])

ru_tok_32k_len = np.array([len(t) for t in ru_tok_32k])
en_tok_32k_len = np.array([len(t) for t in en_tok_32k])

ru_tok_48k_len = np.array([len(t) for t in ru_tok_48k])
en_tok_48k_len = np.array([len(t) for t in en_tok_48k])

for s, f in [('AVG', np.mean), ('MIN', np.min), ('MAX', np.max), ('q0.999', lambda x: np.quantile(x, q=0.999))]:
    print(f''' {s}
             Ru:                  En:
Symbols:        {f(ru_symbol_len):.4f}     \t     {f(en_symbol_len):.4f}
Tok_8k:         {f(ru_tok_8k_len):.4f}     \t     {f(en_tok_8k_len):.4f}
Tok_16k:        {f(ru_tok_16k_len):.4f}    \t     {f(en_tok_16k_len):.4f}
Tok_32k:        {f(ru_tok_32k_len):.4f}    \t     {f(en_tok_32k_len):.4f}
Tok_48k:        {f(ru_tok_48k_len):.4f}    \t     {f(en_tok_48k_len):.4f}

''')

 AVG
             Ru:                  En:
Symbols:        30.7999     	     31.3712
Tok_8k:         8.8042     	     9.0857
Tok_16k:        7.9527    	     8.6329
Tok_32k:        7.4306    	     8.3837
Tok_48k:        7.2365    	     8.3002


 MIN
             Ru:                  En:
Symbols:        2.0000     	     3.0000
Tok_8k:         1.0000     	     2.0000
Tok_16k:        1.0000    	     2.0000
Tok_32k:        1.0000    	     2.0000
Tok_48k:        1.0000    	     2.0000


 MAX
             Ru:                  En:
Symbols:        841.0000     	     1127.0000
Tok_8k:         304.0000     	     345.0000
Tok_16k:        273.0000    	     319.0000
Tok_32k:        246.0000    	     285.0000
Tok_48k:        231.0000    	     263.0000


 q0.999
             Ru:                  En:
Symbols:        169.0000     	     170.0000
Tok_8k:         58.0000     	     55.0000
Tok_16k:        50.0000    	     49.0000
Tok_32k:        44.0000    	     44.0000
Tok_48k:        41.0000    	     42.0

In [11]:
def get_X_y_by_quantile(tokenizer, X, y, q_X, q_y, add_bos_eos_token=False):
    X_t = [tokenizer.encode(t, out_type=int) for t in dataset[1]]
    y_t = [tokenizer.encode(t, out_type=int)for t in dataset[3]]

    X_t_lens = np.array([len(t) for t in X_t])
    y_t_lens = np.array([len(t) for t in y_t])

    X_q = np.quantile(X_t_lens, q=q_X)
    y_q = np.quantile(y_t_lens, q=q_y)

    X_b = X_t_lens <= X_q
    y_b = y_t_lens <= y_q
    mask = X_b & y_b

    X_out = [e for i, e in enumerate(X_t) if mask[i]]

    if add_bos_eos_token:
        y_out = [[tokenizer.bos_id(), *e, tokenizer.eos_id()] for i, e in enumerate(y_t) if mask[i]]
    else:
        y_out = [e for i, e in enumerate(y_t) if mask[i]]

    return X_out, y_out

In [12]:
X_data, y_data = get_X_y_by_quantile(tokenizer16k, dataset[1], dataset[3], 0.999, 0.999, add_bos_eos_token=True)

In [13]:
tokenizer16k.decode(X_data[2]), tokenizer16k.decode(y_data[2])

('Мне пора идти спать.', 'I have to go to sleep.')

In [14]:
max(map(len, X_data)), max(map(len, y_data))

(50, 51)

# Dataset

Since the dataset is rather large, you can omit the validation dataset and just use a set of test sentences after the training.

Create a dataset that returns the following
* A pair of tensors `((None, L), (None, P))` -- input sequence of tokens and output sequence of tokens to be fed into decoder (this should start with the BOS token)
* A tensor `(None, P)` -- output sequence of tokens to be predicted (this should end with EOS token)

In [15]:
class DefaultDataset(keras.utils.PyDataset):
    def __init__(self, X, y, tokenizer, shuffle=False, batch_size=128, q_X=0.999, q_y=0.999, seed=1, **kwargs):
        super().__init__(**kwargs)
        self.tokenizer = tokenizer
        self.X = X
        self.y = y
        self.shuffle = shuffle
        self.batch_size = batch_size
        self.rng = np.random.default_rng(seed)

        self.X, self.y = get_X_y_by_quantile(tokenizer, self.X, self.y, q_X, q_y, add_bos_eos_token=True)
        self.idx = np.arange(len(self.X))
        self.X_lens = np.array([len(t) for t in self.X], dtype=int)
        self.y_lens = np.array([len(t) for t in self.y], dtype=int)

        self.on_epoch_end()

    def __len__(self):
        return -(-len(self.X) // self.batch_size)

    def __getitem__(self, ind):
        index = self.idx[ind * self.batch_size: (ind + 1) * self.batch_size]

        seq = [self.X[i] + self.y[i] for i in index]
        max_l = np.max(self.X_lens[index] + self.y_lens[index])

        sequences = keras.utils.pad_sequences(
            seq, 
            maxlen=max_l,
            padding='post',
            truncating='post',
            value=self.tokenizer.pad_id()
        )

        cols = np.arange(sequences.shape[1]).reshape(1, -1)
        loss_mask = ((cols > self.X_lens[index].reshape(-1, 1)) & (cols < (self.X_lens[index] + self.y_lens[index]).reshape(-1, 1))).astype('float32')

        return sequences[:, :-1], sequences[:, 1:], loss_mask[:, 1:]

    def on_epoch_end(self):
        if self.shuffle:
            self.rng.shuffle(self.idx)


In [16]:
train_dataset = DefaultDataset(X_data, y_data, tokenizer16k, batch_size=3, shuffle=True)

In [17]:
a, b, c = train_dataset[0]

In [18]:
i = 2
print(a[i], b[i], c[i], sep='\n')

[1881 1854 1163  280 2688  317 1137  463  691  260    2 1194 1023  358
 1126  613  664 1446  260    3    0    0    0    0    0    0    0    0
    0    0    0    0    0    0]
[1854 1163  280 2688  317 1137  463  691  260    2 1194 1023  358 1126
  613  664 1446  260    3    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0]
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 1. 1. 1. 1. 1. 1. 1. 1. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


In [19]:
tokenizer16k.decode(a[i])

'Наш план пошёл наперекосяк. Our plan has gone askew.'

# Model

Create a model for training. The model should have two inputs: input sequence `(None, L)` and output sequence`(None, P)`. The model output is a single tensor `(None, P)` logits (or probabilities) of the next token predicted for each input one. The model MUST have decoder only architecture.

In [20]:
@keras.saving.register_keras_serializable()
class EmbeddingAndPositionalEncoding(keras.layers.Layer):
    def __init__(self, vocab_size: int, embed_size: int, max_len: int, **kwargs):
        super().__init__(**kwargs)
        self.vocab_size = vocab_size
        self.embed_size = embed_size
        self.max_len = max_len

        self.embedding_layer = keras.layers.Embedding(vocab_size, embed_size)
        self.positional_embedding_layer = keras.layers.Embedding(max_len, embed_size)

    def call(self, x):
        pos = keras.ops.arange(0, keras.ops.shape(x)[1], 1)

        x_emb = self.embedding_layer(x)
        pos_emb = self.positional_embedding_layer(pos)

        return x_emb + pos_emb

In [21]:
@keras.saving.register_keras_serializable()
class SwiGLU(keras.layers.Layer):
    def __init__(self, embed_size: int, ff_dim: int, **kwargs):
        super().__init__(**kwargs)

        self.gate_proj = keras.layers.Dense(ff_dim, use_bias=False)
        self.up_proj = keras.layers.Dense(ff_dim, use_bias=False)
        self.down_proj = keras.layers.Dense(embed_size, use_bias=False)

    def call(self, x):
        gate = keras.activations.swish(self.gate_proj(x))
        up = self.up_proj(x)
        return self.down_proj(gate * up)

In [22]:
@keras.saving.register_keras_serializable()
class TransformerBlock(keras.layers.Layer):
    def __init__(self, embed_size, num_heads, ff_dim, dropout, **kwargs):
        super().__init__(**kwargs)
        self.embed_size = embed_size
        self.num_heads = num_heads
        self.ff_dim = ff_dim
        self.dropout = dropout

        self.layer_norm1 = keras.layers.LayerNormalization()
        self.Mha = keras.layers.MultiHeadAttention(num_heads, embed_size // num_heads, dropout=dropout)
        self.dropout1 = keras.layers.Dropout(dropout)

        self.layer_norm2 = keras.layers.LayerNormalization()
        self.swiglu = SwiGLU(embed_size, ff_dim)
        self.dropout2 = keras.layers.Dropout(dropout)

    def call(self, x, attention_mask=None, training=False):
        x_mha = self.layer_norm1(x)
        x_mha = self.Mha(x_mha, x_mha, attention_mask=attention_mask, use_causal_mask=True, training=training)

        x_mha = self.dropout1(x_mha, training=training)
        x = x + x_mha

        x_ff = self.layer_norm2(x)
        x_ff = self.swiglu(x_ff)
        x_ff = self.dropout2(x_ff, training=training)

        return x + x_ff


In [23]:
@keras.saving.register_keras_serializable()
class Transformer(keras.Model):
    def __init__(self, vocab_size: int, embed_size: int, max_len: int, num_blocks: int, num_heads: int, ff_dim: int, dropout: float, pad_id: int, name: str, **kwargs):
        super().__init__(name=name, **kwargs)

        self.vocab_size = vocab_size
        self.embed_size = embed_size
        self.max_len = max_len
        self.num_blocks = num_blocks
        self.num_heads = num_heads
        self.ff_dim = ff_dim
        self.dropout = dropout
        self.pad_id = pad_id
        self.name = name

        self.emb_and_pos_layer = EmbeddingAndPositionalEncoding(vocab_size, embed_size, max_len, name='emb_and_pos')

        self.transformer_blocks = [TransformerBlock(embed_size, num_heads, ff_dim, dropout, name=f'MHA_{i}') for i in range(num_blocks)]

        self.layer_norm = keras.layers.LayerNormalization()

        self.next_token_classifyer = keras.layers.Dense(vocab_size)
        

    def call(self, x, training=False):
        attention_mask = keras.ops.not_equal(x, self.pad_id)[:, None, :]

        x = self.emb_and_pos_layer(x)

        for trans_bl in self.transformer_blocks:
            x = trans_bl(x, attention_mask=attention_mask, training=training)

        x = self.layer_norm(x)
        x = self.next_token_classifyer(x)

        return x


    def get_config(self):
        config = super().get_config()
        config.update({
            'vocab_size': self.vocab_size,
            'embed_size': self.embed_size,
            'max_len': self.max_len,
            'num_blocks': self.num_blocks,
            'num_heads': self.num_heads,
            'ff_dim': self.ff_dim,
            'dropout': self.dropout,
            'pad_id': self.pad_id,
        })
        return config


In [24]:
max(map(len, X_data)), max(map(len, y_data))

(50, 51)

In [25]:
MAX_LEN = 101

In [26]:
model = Transformer(
    tokenizer16k.vocab_size(),
    128,
    MAX_LEN,
    6,
    8,
    256,
    0.05,
    tokenizer16k.pad_id(),
    name='model'
)

In [27]:
model(keras.ops.empty((3, 20)))

<tf.Tensor: shape=(3, 20, 16000), dtype=float32, numpy=
array([[[-0.07974865, -0.15735216, -0.00136017, ...,  0.0308469 ,
         -0.22972403, -0.07217318],
        [-0.25913218, -0.08653687,  0.22161976, ..., -0.01653389,
         -0.0613237 , -0.09360041],
        [ 0.1424826 ,  0.29642448, -0.0841249 , ..., -0.03075811,
          0.14828996,  0.03954979],
        ...,
        [-0.11945116,  0.13675162,  0.01376192, ...,  0.14289753,
          0.26785275, -0.05887818],
        [-0.05123598, -0.13607891,  0.04951836, ...,  0.15720873,
         -0.0653723 , -0.16410512],
        [ 0.07832248,  0.16516773,  0.06902177, ..., -0.04429069,
          0.07779091, -0.07317751]],

       [[-0.07974865, -0.15735216, -0.00136017, ...,  0.0308469 ,
         -0.22972403, -0.07217318],
        [-0.25913218, -0.08653687,  0.22161976, ..., -0.01653389,
         -0.0613237 , -0.09360041],
        [ 0.1424826 ,  0.29642448, -0.0841249 , ..., -0.03075811,
          0.14828996,  0.03954979],
        ...

In [28]:
model.summary()

Model: "model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ emb_and_pos                     │ ?                      │     2,060,928 │
│ (EmbeddingAndPositionalEncodin… │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ MHA_0 (TransformerBlock)        │ ?                      │       164,864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ MHA_1 (TransformerBlock)        │ ?                      │       164,864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ MHA_2 (TransformerBlock)        │ ?                      │       164,864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ MHA_3 (TransformerBlock)        │ ?                      │       164,864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ MHA_4 (TransformerBlock)        │ ?                      │       164,864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ MHA_5 (TransformerBlock)        │ ?                      │       164,864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ layer_normalization_12          │ (3, 20, 128)           │           256 │
│ (LayerNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_18 (Dense)                │ (3, 20, 16000)         │     2,064,000 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,114,368 (19.51 MB)

 Trainable params: 5,114,368 (19.51 MB)

 Non-trainable params: 0 (0.00 B)

# Training

Train your model using teacher forcing.

In [29]:
X_train, X_val, y_train, y_val = train_test_split(dataset[1], dataset[3], test_size=0.05)

In [30]:
train_dataset = DefaultDataset(X_train, y_train, tokenizer16k, shuffle=True, batch_size=16)
val_dataset = DefaultDataset(X_val, y_val, tokenizer16k, shuffle=False, batch_size=16)

In [31]:
EPOCHS = 5

In [32]:
STEPS_PER_EPOCH = len(train_dataset)
TOTAL_STEPS = STEPS_PER_EPOCH * EPOCHS

WARMUP_RATIO = 0.1
WARMUP_STEPS = int(TOTAL_STEPS * WARMUP_RATIO)

cosine_decay = keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=0.0,
    warmup_target=1e-3,
    warmup_steps=WARMUP_STEPS,
    decay_steps=TOTAL_STEPS - WARMUP_STEPS,
    alpha=0.1
)

model.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=keras.optimizers.AdamW(learning_rate=cosine_decay)
)

In [33]:
SAVE_DIR = 'models'

def make_callbacks(name):
    return [
        keras.callbacks.ModelCheckpoint(
            filepath=os.path.join(SAVE_DIR, f'{name}.keras'),
            monitor='val_loss',
            mode='min',
            save_best_only=True,
            save_weights_only=False,
            verbose=1
        ),

        keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=10,
            mode='min',
            restore_best_weights=True,
            verbose=1
        )
    ]

In [208]:
model.fit(train_dataset, validation_data=val_dataset, epochs=EPOCHS, callbacks=make_callbacks('model'))

Epoch 1/5
   10/51861 ━━━━━━━━━━━━━━━━━━━━ 2:43:28 189ms/step - loss: 2.9164

W0000 00:00:1795168158.729426   72113 cpu_allocator_impl.cc:82] Allocation of 77824000 exceeds 10% of free system memory.


   16/51861 ━━━━━━━━━━━━━━━━━━━━ 2:58:31 207ms/step - loss: 3.0353

W0000 00:00:1795168160.089936   72113 cpu_allocator_impl.cc:82] Allocation of 55296000 exceeds 10% of free system memory.


   22/51861 ━━━━━━━━━━━━━━━━━━━━ 2:56:52 205ms/step - loss: 2.9826

W0000 00:00:1795168161.049596   72115 cpu_allocator_impl.cc:82] Allocation of 44032000 exceeds 10% of free system memory.


... (2 outputs left out)

   45/51861 ━━━━━━━━━━━━━━━━━━━━ 2:49:46 197ms/step - loss: 3.0398

W0000 00:00:1795168165.382258   72118 cpu_allocator_impl.cc:82] Allocation of 48128000 exceeds 10% of free system memory.


  212/51861 ━━━━━━━━━━━━━━━━━━━━ 2:51:40 199ms/step - loss: 3.0043

KeyboardInterrupt: 

# Testing

Make a function for text translation. Translate some text and evaluate model performance.

In [42]:
model = keras.models.load_model('models/model_16k_128_101_8_12_256_ep8.keras')

In [43]:
model.summary()

Model: "model_16k_128_101_8_12_256"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ emb_and_pos                     │ ?                      │     2,060,928 │
│ (EmbeddingAndPositionalEncodin… │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ MHA_0 (TransformerBlock)        │ ?                      │       160,744 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ MHA_1 (TransformerBlock)        │ ?                      │       160,744 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ MHA_2 (TransformerBlock)        │ ?                      │       160,744 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ MHA_3 (TransformerBlock)        │ ?                      │       160,744 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ MHA_4 (TransformerBlock)        │ ?                      │       160,744 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ MHA_5 (TransformerBlock)        │ ?                      │       160,744 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ MHA_6 (TransformerBlock)        │ ?                      │       160,744 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ MHA_7 (TransformerBlock)        │ ?                      │       160,744 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ layer_normalization_114         │ (256, 68, 128)         │           256 │
│ (LayerNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_168 (Dense)               │ (256, 68, 16000)       │     2,064,000 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 16,233,409 (61.93 MB)

 Trainable params: 5,411,136 (20.64 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 10,822,273 (41.28 MB)

In [44]:
def translate(
    text: str,
    tokenizer: 'tokenizer',
    model: keras.Model,
    max_len: int = 20
) -> str:
    '''Predicts `text`translation using the `model`.

    Arguments:
        text: text to be translated
        tokenizer: tokenizer to use
        model: model ot use
        max_len: maximum length of the prediction (in tokens)

    Returns:
        tranlated text'''
    ...

    tok = tokenizer.encode(text) + [tokenizer.bos_id()]
    predicted = []

    loc_tok = -1
    while loc_tok != tokenizer.eos_id() and len(tok) < MAX_LEN:
        loc_tok = model(np.array(tok)[None])[0][-1]
        loc_tok = int(keras.ops.argmax(loc_tok))
        tok.append(loc_tok)
        predicted.append(loc_tok)

    return tokenizer.decode(predicted, out_type=str)


In [45]:
test_sentences = [
    "ываыв оорлр укцкукцук мсисмимрпо ропр",
    "Абракадабра",

    "Привет, как дела?",
    "Сколько это стоит?",
    "Я стреляю из лука.",
    "Мне нравится изучать иностранные языки.",
    "Этот код потребляет слишком много памяти.",
    "Я живу в Москве уже десять лет.",
    "Она не пришла на работу, потому что заболела.",
    "Если пойдёт дождь, мы останемся дома и посмотрим фильм.",
    "Мне пришлось отменить встречу, потому что я опоздал на поезд.",
    "Без труда не вытащишь и рыбку из пруда.",
    "Я проснулся ни свет ни заря, чтобы успеть на первый поезд.",
    "Финансовый директор предупредил, что бюджет будет сокращён в следующем квартале.",
    "- Ты придёшь? - Вряд ли.",
    "Дом был построен в 1890 году.",
    "Ключ к успеху - терпение.",
    "Не делай из мухи слона.",
    "Илон Маск посетил Москву.",
]

for text in test_sentences:
    translation = translate(text, tokenizer16k, model, MAX_LEN)
    print(f"RU:                 {text}")
    print(f"EN model:           {translation}")

RU:                 ываыв оорлр укцкукцук мсисмимрпо ропр
EN model:           The church windscans the wind of acrestic.
RU:                 Абракадабра
EN model:           Abradadada is Robragorican.
RU:                 Привет, как дела?
EN model:           Hi, how are you?
RU:                 Сколько это стоит?
EN model:           How much does it cost?
RU:                 Я стреляю из лука.
EN model:           I'm shooting the onions.
RU:                 Мне нравится изучать иностранные языки.
EN model:           I like learning foreign languages.
RU:                 Этот код потребляет слишком много памяти.
EN model:           This code is too much use to read.
RU:                 Я живу в Москве уже десять лет.
EN model:           I've lived in Moscow for ten years.
RU:                 Она не пришла на работу, потому что заболела.
EN model:           She didn't come to work because she was sick.
RU:                 Если пойдёт дождь, мы останемся дома и посмотрим фильм.
EN model: 

In [46]:
translate('Я стреляю из лука.', tokenizer16k, model, MAX_LEN)

"I'm shooting the onions."

# Bonus

Train the model using its own predictions as inputs. The probability to use the predicted tokens instead of the ones from the dataset should increase as the training progresses. Train for one or two epochs. Evaluate the model, does the model perform better?

In [47]:
model_train = keras.models.load_model('models/model_16k_128_101_8_12_256_ep8.keras')

In [48]:
train_dataset = DefaultDataset(dataset[1], dataset[3], tokenizer16k, shuffle=True, batch_size=16)

In [49]:
len(train_dataset[0])

3

In [50]:
EPOCHS = 3

STEPS_PER_EPOCH = len(train_dataset)
TOTAL_STEPS = STEPS_PER_EPOCH * EPOCHS

WARMUP_RATIO = 0.1
WARMUP_STEPS = int(TOTAL_STEPS * WARMUP_RATIO)

cosine_decay = keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=0.0,
    warmup_target=1e-3,
    warmup_steps=WARMUP_STEPS,
    decay_steps=TOTAL_STEPS - WARMUP_STEPS,
    alpha=0.1
)

optimizer = keras.optimizers.AdamW(learning_rate=cosine_decay)
loss = keras.losses.SparseCategoricalCrossentropy(from_logits=True, reduction='none')

In [51]:
train_dataset[0][2]

array([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0.],
       [0., 0., 0., 0., 0., 0., 1., 1., 1., 1., 1., 1., 1., 1., 1., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0.],
       [0., 0., 0., 0., 0., 1., 1., 1., 1., 1., 1., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0.],
       [0., 0., 0., 0., 0., 1., 1.

In [52]:
def train_step_wrapper(model, optimizer, loss, eos_token_id, pad_token_id):
    @tf.function(reduce_retracing=True)
    def train_step(ds, prob):
        X = ds[0]
        y = ds[1]
        loss_mask = ds[2]

        batch_size = keras.ops.shape(X)[0]

        X = keras.ops.concatenate([X, y[:, -1][:, None]], axis=1)

        start_idx = int(keras.ops.argmax(keras.ops.sum(loss_mask, axis=0) > 0.)) + 1

        X_new = X[:, :start_idx]

        for i in tf.range(start_idx, tf.shape(X)[1]):
            X_pred = keras.ops.argmax(model(X_new, training=False)[:, -1], axis=-1)[:, None]
            X_true = X[:, i][:, None]

            r = keras.random.uniform((batch_size, 1))
            X_add2new = keras.ops.where((r > prob) | (X_true == eos_token_id) | (X_true == pad_token_id), X_true, X_pred)

            X_new = keras.ops.concatenate([X_new, X_add2new], axis=1)

        with tf.GradientTape() as tape:
            pred = model(X_new[:, :-1], training=True)
            loss_val = tf.reduce_sum(loss(X[:, 1:], pred) * loss_mask) / tf.reduce_sum(loss_mask)
            
        gradients = tape.gradient(
            loss_val,
            model.trainable_variables
        )

        optimizer.apply_gradients(
            zip(gradients, model.trainable_variables)
        )

        return loss_val
        
    return train_step

In [101]:
tr = train_step_wrapper(model_train, optimizer, loss, tokenizer16k.eos_id(), tokenizer16k.pad_id())

steps = 0
for epoch in range(EPOCHS):
    for b in tqdm(train_dataset, desc=f'Эпоха {epoch}'):
        tr(b, steps / (TOTAL_STEPS - 1))

        steps += 1

    print(model.evaluate(val_dataset))
    model.save(f'models/model_16k_128_101_8_12_256_ep8_finetuned_ep{epoch}.keras')

Эпоха 0:   0%|          | 0/51861 [00:12<?, ?it/s]


KeyboardInterrupt: 

In [53]:
modelep8ep0 = keras.models.load_model('models/model_16k_128_101_8_12_256_ep8.keras')
modelep8ep1 = keras.models.load_model('models/model_16k_128_101_8_12_256_ep8_finetuned_ep1.keras')
modelep8ep2 = keras.models.load_model('models/model_16k_128_101_8_12_256_ep8_finetuned_ep2.keras')
modelep8ep3 = keras.models.load_model('models/model_16k_128_101_8_12_256_ep8_finetuned_ep2.keras')

In [54]:
for text in test_sentences:
    print(f"RU:                 {text}")
    print(f"EN model 8ep:       {translate(text, tokenizer16k, modelep8ep0, MAX_LEN)}")
    print(f"EN model 8ep 1ep:   {translate(text, tokenizer16k, modelep8ep1, MAX_LEN)}")
    print(f"EN model 8ep 2ep:   {translate(text, tokenizer16k, modelep8ep2, MAX_LEN)}")
    print(f"EN model 8ep 3ep:   {translate(text, tokenizer16k, modelep8ep3, MAX_LEN)}\n")

RU:                 ываыв оорлр укцкукцук мсисмимрпо ропр
EN model 8ep:       The church windscans the wind of acrestic.
EN model 8ep 1ep:   The church windows are squeaking by a carpoway.
EN model 8ep 2ep:   The knights a scubs a surbs.
EN model 8ep 3ep:   The knights a scubs a surbs.

RU:                 Абракадабра
EN model 8ep:       Abradadada is Robragorican.
EN model 8ep 1ep:   Abradada is Road.
EN model 8ep 2ep:   Abradadadada is.
EN model 8ep 3ep:   Abradadadada is.

RU:                 Привет, как дела?
EN model 8ep:       Hi, how are you?
EN model 8ep 1ep:   Hi, how are you?
EN model 8ep 2ep:   Hi, how are you doing?
EN model 8ep 3ep:   Hi, how are you doing?

RU:                 Сколько это стоит?
EN model 8ep:       How much does it cost?
EN model 8ep 1ep:   How much does it cost?
EN model 8ep 2ep:   How much does this cost?
EN model 8ep 3ep:   How much does this cost?

RU:                 Я стреляю из лука.
EN model 8ep:       I'm shooting the onions.
EN model 8ep 1ep:   